# Main CW

## Objective: Deploy ML Pipeline for image classification task employing a CNN model using TensorFlow and Keras.

In [ ]:
!pip install tensorflow 
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.datasets import cifar10

In [ ]:
def create_model_v1():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Conv2D(128, (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
def create_model_v2():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.4),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.0008), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
def create_model_v3():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        Conv2D(32, (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Conv2D(128, (3,3), activation='relu', padding='same'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.3),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
# Load CIFAR-10 dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
print(X_train.shape, X_test.shape)  # (50000, 32, 32, 3) (10000, 32, 32, 3)
# Normalize pixel values to range [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
# Convert class labels to one-hot encoded vectors
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)
# Split the training data into training and validation sets (80/20 split)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
# === EDA: Class Distribution and Sample Inspection ===
import collections

# Check label distribution in training data
label_counts = collections.Counter(np.argmax(y_train, axis=1))
plt.figure(figsize=(8, 5))
ax = sns.barplot(x=list(label_counts.keys()), y=list(label_counts.values()))
plt.title("Class Distribution in Training Data")
plt.xlabel("Class Index")
plt.ylabel("Sample Count")
plt.ylim(3800, max(label_counts.values()) + 100)  # Set y-axis start at 3000


plt.tight_layout()
plt.show()




In [ ]:
# Setup early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train models
model1 = create_model_v1()
history1 = model1.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, batch_size=64, callbacks=[early_stop])

model2 = create_model_v2()
history2 = model2.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, batch_size=64, callbacks=[early_stop])

model3 = create_model_v3()
history3 = model3.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, batch_size=64, callbacks=[early_stop])


In [ ]:
def compare_histories(histories, labels):
    plt.figure(figsize=(12,5))

    # Accuracy comparison
    plt.subplot(1,2,1)
    for history, label in zip(histories, labels):
        plt.plot(history.history['val_accuracy'], label=f'{label} Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Loss comparison
    plt.subplot(1,2,2)
    for history, label in zip(histories, labels):
        plt.plot(history.history['val_loss'], label=f'{label} Val Loss')
    plt.title('Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

compare_histories([history1, history2, history3], ['Model 1', 'Model 2', 'Model 3'])


In [ ]:
# Get test accuracies
test_scores = {}
for model, label in zip([model1, model2, model3], ['Model 1', 'Model 2', 'Model 3']):
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    test_scores[label] = acc
    print(f'{label} Test Accuracy: {acc:.4f}')


In [ ]:
# === Detailed Evaluation of Best Model (Model 3) ===
from sklearn.metrics import classification_report, confusion_matrix

# Predict on test set using Model 3
y_pred = model3.predict(X_test)
y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

# Classification Report
print("Classification Report for Model 3:\n")
print(classification_report(y_true_labels, y_pred_labels, target_names=class_names))

# Confusion Matrix
conf_matrix = confusion_matrix(y_true_labels, y_pred_labels)
plt.figure(figsize=(10,8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix for Model 3')
plt.show()


In [ ]:
def show_predictions(model, X_test, y_test, class_names):
    y_probs = model.predict(X_test[:5])
    y_preds = np.argmax(y_probs, axis=1)
    y_true = np.argmax(y_test[:5], axis=1)

    plt.figure(figsize=(12,5))
    for i in range(5):
        plt.subplot(1, 5, i+1)
        plt.imshow(X_test[i])
        plt.axis('off')
        plt.title(f"True: {class_names[y_true[i]]}\nPred: {class_names[y_preds[i]]}\nConf: {y_probs[i][y_preds[i]]:.2f}")
    plt.show()

# Example usage
class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
show_predictions(model3, X_test, y_test, class_names)
